In [34]:
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np

In [55]:
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

def list_redundant(lst):
 
    ele = lst[0]
    chk = True
 
    # Comparing each element with first item
    for item in lst:
        if ele != item:
            chk = False
            break
        
    return chk

In [24]:
feature_table=pd.read_csv('rack_01--well_00--roi_02--exp_level_01_backsub--cellpose_expanded_d08_whole_cell.csv')
signat_matrix=pd.read_csv('signature_matrix.csv')
img_data={
        'Cell':feature_table['CellID'].values,
        'X':feature_table['X_centroid'].values,
        'Y':feature_table['Y_centroid'].values
        }

In [25]:
MOI=signat_matrix.columns[2::]
MOI

Index(['CD3', 'CD66b', 'Cytokeratin', 'Mast_Cell_Tryptase', 'Actin',
       'Myosin_Smooth_Muscle', 'CD20', 'CD56', 'CD45RO', 'CD138', 'CD68',
       'CD14', 'CD8', 'CD4', 'FOXP3', 'CD31'],
      dtype='object')

In [57]:
name_map={}

for marker in MOI:
    
    scores=[]
    marker_in_feat=[]
    
    for c in feature_table.columns.values:
      
        similarity=similar(marker,c)
        
        if similarity > 0.8:
            #print('{M}:{C}:score={S}'.format(M=marker,C=c,S=similarity))
            scores.append(similarity)
            marker_in_feat.append(c)
        
        
        if len(scores)>0:
            if (list_redundant(scores) and len(scores)!=1 ):
                
                name_map[marker]=marker_in_feat
                
            else:
                
                name_map[marker]=marker_in_feat[np.argmax(scores)]
        
        
name_map      

{'CD3': 'CD3',
 'CD66b': 'CD66b',
 'Cytokeratin': 'Cytokeratin',
 'Mast_Cell_Tryptase': 'Mast_Cell_Tryptase',
 'Actin': 'Actin',
 'Myosin_Smooth_Muscle': 'Myosin_Smooth_Muscle',
 'CD20': 'CD20',
 'CD56': 'CD56',
 'CD45RO': 'CD45RO',
 'CD138': 'CD138',
 'CD68': 'CD68',
 'CD14': 'CD14',
 'CD8': ['CD68', 'CD8a', 'CD88'],
 'CD4': 'CD4',
 'FOXP3': 'FOXP3',
 'CD31': 'CD31'}

In [60]:
name_map['CD8']='CD8a'

In [62]:
for key,val in name_map.items():
    img_data[key]=feature_table[val].values

In [68]:
img_data_df=pd.DataFrame(img_data)
img_data_df.to_csv('img_data.csv',index=False)